# Traveller System Builder

Generates a single star system in detail using the `worldmaker` package,
following the Mongoose Traveller 2e *World Builder's Handbook*.

This notebook is a thin front-end: all the rules live in the package, so it
never drifts from the tested implementation. Run the cells in order.


In [ ]:
import random

import worldmaker as wm

# Set a seed for a reproducible system, or comment out for a fresh one
random.seed(1105)

system = wm.generate_full_system("Example System")
print(system.name, "|", len(system.stars), "stars |", system.total_worlds, "worlds")
print("Age:", system.age_gyr, "Gyr")
print("Profile:", wm.generator.generate_short_profile(system))

## Stars

In [ ]:
for star in system.stars:
    if star.is_composite:
        continue
    print(f"{star.designation:4s} {star.spectral_type:10s} "
          f"mass {star.mass:6.3f}  L {star.luminosity:9.5f}  "
          f"HZCO {star.hzco:5.2f}  MAO {star.mao:5.3f}")
    if star.orbit_class:
        print(f"      {star.orbit_class} orbit {star.orbit_num} "
              f"ecc {star.eccentricity} period {star.period_years} yr")

## Worlds\n\nEvery body in orbital order, with its physical characteristics.

In [ ]:
df = wm.create_system_dataframe(system)
df

## The mainworld in full

In [ ]:
mw = system.mainworld
print("UWP           ", mw.uwp)
print("Trade codes   ", ", ".join(mw.trade_codes))
print("Habitability  ", mw.habitability_rating)
print("Temperature   ", f"{mw.low_temperature} / {mw.mean_temperature} / "
      f"{mw.high_temperature} K (low / mean / high)")
print("Atmosphere    ", mw.atmosphere_name, f"{mw.atmos_pressure_bar} bar",
      f"ppO2 {mw.partial_pressure_oxygen}")
print("Seismology    ", mw.seismic_activity, f"({mw.tectonic_plates} plates)")
print("Population    ", mw.population_profile, f"= {mw.total_population:,} people")
print("Technology    ", mw.technology_profile)
print("Economics     ", mw.economic_extension, f"WTN {mw.wtn}",
      f"GWP/capita Cr{mw.gwp_per_capita:,.0f}")
print("Government    ", mw.government_profile, mw.government_type)
print("Law           ", mw.law_profile, "justice", mw.justice_profile)
print("Military      ", mw.military_profile)
print("Life          ", mw.native_lifeform_profile, "-", mw.life_details)

## Major cities

In [ ]:
for city in mw.major_cities:
    port = f" (port {city['starport']})" if city['starport'] else ""
    capital = " [capital]" if city.get('is_capital') else ""
    print(f"{city['name']:20s} {city['population']:>14,}{port}{capital}")

## Satellites\n\nMoon orbits are in planetary diameters (PD) from the primary.

In [ ]:
for world in system.all_worlds:
    if not world.satellites:
        continue
    print(f"{world.designation} ({world.body_type}, size {world.size_code})")
    for sat in world.satellites:
        kind = "ring" if sat.is_ring else f"size {sat.size_code}"
        print(f"    {sat.designation:12s} {kind:10s} "
              f"orbit {sat.orbit_pd:7.2f} PD  period {sat.period_hours:9.1f} h")

## Full dossier

In [ ]:
print(wm.export_system_markdown(system))

## World surface map

In [ ]:
from IPython.display import SVG, display

terrain = wm.generate_world_terrain(mw)
print("Terrain coverage:", wm.terrain_summary(terrain))
display(SVG(wm.render_world_map_svg(mw, terrain)))